# 01 – Data exploration

**Goal:** understand the structure of each raw file before cleaning it.

For every file I check:
- which sheet holds the data,
- which row holds the column headers,
- which column holds the area code (needed to join the tables),
- the time period covered and any special values such as `[x]` (not available) or `[z]` (not applicable).

Raw files are described in `data/raw/SOURCES.md`. Nothing in `data/raw/` is modified.

*Note: the large `.ods` vehicle files take about 1–2 minutes each to load.*

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 200)

RAW = "../data/raw/"

## 1. Licensed vehicles by fuel type and local authority (VEH0105)

All licensed vehicles, quarterly, by body type, fuel type, keepership and area. Headers are on row 5 of the sheet (`header=4`, because Python counts from 0).

In [2]:
veh0105 = pd.read_excel(RAW + "veh0105.ods", sheet_name="VEH0105", header=4)
print(veh0105.shape)
veh0105.head()

(40759, 74)


,Units,BodyType,Fuel,Keepership,ONS Sort,ONS Code,ONS Geography,...,2011 Q2,2011 Q1,2010 Q4,2010 Q3,2010 Q2,2010 Q1,2009 Q4
0,Thousands,Buses and coaches,Diesel,Company,1,K02000001,United Kingdom,...,[x],[x],[x],[x],[x],[x],[x]
1,Thousands,Buses and coaches,Diesel,Company,2,K03000001,Great Britain,...,132.245,132.133,132.315,133.697,133.139,133.375,133.488
2,Thousands,Buses and coaches,Diesel,Company,3,E92000001,England,...,111.141,110.934,111.136,112.243,111.813,111.896,112.022
3,Thousands,Buses and coaches,Diesel,Company,4,E12000001,North East,...,5.289,5.253,5.341,5.42,5.377,5.376,5.351
4,Thousands,Buses and coaches,Diesel,Company,5,E06000047,County Durham,...,1.513,1.47,1.521,1.521,1.477,1.47,1.497


In [3]:
# Category columns and their values
for col in ["Units", "BodyType", "Fuel", "Keepership"]:
    print(col, ":", veh0105[col].unique().tolist())

Units : ['Thousands']
BodyType : ['Buses and coaches', 'Cars', 'Heavy goods vehicles', 'Light goods vehicles', 'Motorcycles', 'Other vehicles', 'Total']
Fuel : ['Diesel', 'Hybrid electric (petrol)', 'Other fuel types', 'Petrol', 'Total']
Keepership : ['Company', 'Private', 'Total']


In [4]:
# Quarter columns run from newest to oldest
quarters = veh0105.columns[7:]
print("Latest quarter:", quarters[0], "| Earliest quarter:", quarters[-1], "| Number of quarters:", len(quarters))

Latest quarter: 2026 Q2 | Earliest quarter: 2009 Q4 | Number of quarters: 67


**Observations**
- Values are in **thousands** of vehicles.
- The data is in *wide* format: one column per quarter (2009 Q4 to 2026 Q2). It will need reshaping to *long* format (one row per area and quarter) for analysis.
- **There is no separate "Battery electric" fuel type.** BEVs are inside "Other fuel types". BEV counts must therefore come from VEH0142, and VEH0105 gives the total fleet.

## 2. Licensed plug-in vehicles by local authority (VEH0142)

Same layout as VEH0105, but only plug-in vehicles, split into battery electric and plug-in hybrids.

In [5]:
veh0142 = pd.read_excel(RAW + "veh0142.ods", sheet_name="VEH0142", header=4)
print(veh0142.shape)
for col in ["Units", "BodyType", "Fuel", "Keepership"]:
    print(col, ":", veh0142[col].unique().tolist())

(26156, 74)
Units : ['Number']
BodyType : ['Buses and coaches', 'Cars', 'Heavy goods vehicles', 'Light goods vehicles', 'Motorcycles', 'Other vehicles', 'Total']
Fuel : ['Battery electric', 'Plug-in hybrid electric (diesel)', 'Plug-in hybrid electric (petrol)', 'Range extended electric', 'Total']
Keepership : ['Company', 'Private', 'Total']


In [6]:
# Quick sanity check: battery electric cars in the whole UK
uk_bev_cars = veh0142[
    (veh0142["ONS Code"] == "K02000001")
    & (veh0142["BodyType"] == "Cars")
    & (veh0142["Fuel"] == "Battery electric")
    & (veh0142["Keepership"] == "Total")
]
uk_bev_cars[["ONS Geography", "2026 Q2", "2025 Q2", "2024 Q2", "2023 Q2", "2022 Q2"]]

,ONS Geography,2026 Q2,2025 Q2,2024 Q2,2023 Q2,2022 Q2
2157,United Kingdom,2016968,1504927,1089220,779420,492150


**Observations**
- Values are **actual numbers** (not thousands), unlike VEH0105. Units must be aligned before combining the two tables.
- The UK figure of about 2.0 million battery electric cars in 2026 Q2 is a useful sanity check against published DfT headlines.

### Geography levels in the vehicle tables

The vehicle tables mix several geography levels in one column: UK, countries, regions, counties and local authorities. The first three characters of the ONS code tell us the level.

In [7]:
veh0142["code_prefix"] = veh0142["ONS Code"].astype(str).str[:3]
veh0142.drop_duplicates("ONS Code")["code_prefix"].value_counts()

code_prefix
E07    164
E06     63
E08     36
E09     33
S12     32
W06     22
E10     21
N09     11
E12      9
E11      6
E13      2
[z]      1
K03      1
W92      1
S92      1
E92      1
N92      1
K02      1
Name: count, dtype: int64

In [8]:
# Rows without a real area code
veh0142.loc[veh0142["ONS Code"] == "[z]", "ONS Geography"].str.strip().unique()[:10]

array(['Local Authority unknown within North East',
       'Local Authority unknown within South East',
       'Vehicle under disposal previously GB',
       'Vehicle under disposal previously NI', 'Region or county unknown',
       'Local Authority unknown within East Midlands',
       'District unknown within Tyne and Wear (Met County)',
       'District unknown within Greater Manchester (Met County)',
       'District unknown within Lancashire',
       'District unknown within Merseyside (Met County)'], dtype=object)

**Observations**
- `K02` = UK, `K03` = Great Britain, `E92/W92/S92/N92` = countries, `E12` = English regions, `E10/E11/E13` = counties, metropolitan counties and Inner/Outer London.
- `E06, E07, E08, E09, W06, S12, N09` are the **lower-tier local authorities** used in the charger data. Only these will be kept, otherwise vehicles would be counted twice (once in a county and again in its districts).
- Rows coded `[z]` are vehicles with an unknown area or under disposal. They will be excluded from local analysis.

## 3. Vehicles registered for the first time (VEH1153)

New registrations show the *speed* of change better than the total fleet, because the fleet changes slowly as old cars are replaced.

In [9]:
veh1153 = pd.read_excel(RAW + "veh1153.ods", sheet_name="VEH1153a_RoadUsing", header=4)
print(veh1153.shape)
print("Columns:", veh1153.columns.tolist())
for col in ["Geography", "Date Interval", "Units", "BodyType", "Keepership"]:
    print(col, ":", veh1153[col].unique().tolist())

(44402, 20)
Columns: ['Geography', 'Date Interval', 'Date', 'Units', 'BodyType', 'Keepership', 'Petrol', 'Diesel', 'Hybrid Electric - Petrol', 'Hybrid Electric - Diesel', 'Plug-in Hybrid Electric - Petrol', 'Plug-in Hybrid Electric - Diesel', 'Battery Electric', 'Range Extended Electric', 'Fuel Cell Electric', 'Gas', 'Others', 'Total', 'Plugin', 'ZEV']
Geography : ['England', 'Scotland', 'Wales', 'Great Britain', 'Northern Ireland', 'United Kingdom']
Date Interval : ['Annual', 'Quarterly', 'Monthly']
Units : ['Thousands']
BodyType : ['Buses and coaches', 'Cars', 'Heavy goods vehicles', 'Light goods vehicles', 'Motorcycles', 'Other vehicles', 'Total']
Keepership : ['Company', 'Private', 'Total', 'Disposal']


In [10]:
# New cars in the UK by year: battery electric share
new_cars = veh1153[
    (veh1153["Geography"] == "United Kingdom")
    & (veh1153["Date Interval"] == "Annual")
    & (veh1153["BodyType"] == "Cars")
    & (veh1153["Keepership"] == "Total")
]
new_cars[["Date", "Petrol", "Diesel", "Battery Electric", "Total"]].tail(8)

,Date,Petrol,Diesel,Battery Electric,Total
41155,2018,1489.920,755.625,15.756,2394.04
41156,2019,1540.120,622.089,38.007,2346.58
41157,2020,1007.860,304.443,107.913,1656.40
41158,2021,911.684,194.692,190.420,1677.25
41159,2022,821.258,127.902,266.875,1652.13
41160,2023,898.808,126.170,314.222,1945.83
41161,2024,749.208,110.550,381.869,2000.11
41162,2025,610.053,95.838,473.225,2071.39


**Observations**
- This table is already in *long* format: fuel types are columns, and dates are rows (annual, quarterly and monthly).
- Values are in thousands. Geography is national only (UK, GB and the four nations), not local authority.
- Monthly figures run to June 2026, so the latest full year is 2025.

## 4. Public EV chargers by local authority (EVCI0102)

Sheet `EVCI0102a` holds all public chargers, and `EVCI0102b` holds the 50kW+ (rapid) chargers. Headers are on row 3 (`header=2`).

In [11]:
evci = pd.read_excel(RAW + "evci0102_2026-07_EV_chargers_by_local_authority_UK.ods", sheet_name="EVCI0102a", header=2)
print(evci.shape)
evci.head()

(361, 5)


,Local authority code,Local authority,1 January 2026,1 April 2026,1 July 2026[note 5]
0,E07000223,Adur,103,112,109
1,E07000032,Amber Valley,102,92,98
2,E07000224,Arun,273,278,283
3,E07000170,Ashfield,100,116,133
4,E07000105,Ashford,204,212,204


In [12]:
evci["Local authority code"].str[:3].value_counts()

Local authority code
E07    164
E06     63
E08     36
E09     33
S12     32
W06     22
N09     11
Name: count, dtype: int64

**Observations**
- 361 lower-tier local authorities across the UK, with snapshots for January, April and July 2026.
- The July 2026 column name contains a note reference (`1 July 2026[note 5]`) and should be renamed.

## 5. Historic charger series (EVCI9001)

Sheet `1a` gives chargers by local authority every quarter since October 2019, so we can see whether infrastructure is growing as fast as EV numbers.

In [13]:
evci_hist = pd.read_excel(RAW + "evci9001_2026-07_EV_charging_devices_UK.ods", sheet_name="1a", header=2)
print(evci_hist.shape)
print("Date columns:", evci_hist.columns[2:].tolist())
evci_hist.head()

(433, 30)
Date columns: ['Oct-19', 'Jan-20', 'Apr-20', 'Jul-20', 'Oct-20', 'Jan-21', 'Apr-21', 'Jul-21', 'Oct-21', 'Jan-22', 'Apr-22', 'Jul-22', 'Oct-22', 'Jan-23', 'Apr-23', 'Jul-23', 'Oct-23', 'Jan-24', 'Apr-24', 'Jul-24', 'Oct-24 [Note 14]', 'Jan-25', 'Apr-25', 'Jul-25', 'Oct-25', 'Jan-26', 'Apr-26', 'Jul-26']


,Local authority / region code,Local authority / region name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,...,Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26,Jul-26
0,K02000001,United Kingdom,15116,16505,17947,18265,19487,...,73334,76507,82002,86021,87796,92141,94595
1,K03000001,Great Britain,14821,16210,17642,17953,19169,...,72654,75835,81318,85283,87069,91414,93864
2,E92000001,England,12549,13719,14979,15395,16456,...,63389,65877,70672,74115,75818,79198,81231
3,E12000001,North East,738,752,786,812,849,...,2583,2553,2703,2698,2734,2950,3090
4,E06000047,County Durham,92,96,102,105,106,...,400,462,482,482,489,523,539


**Observations**
- Wide format, with one column per quarter from Oct-19. The first rows are UK, GB, countries and regions, followed by local authorities, so the same filtering as in the vehicle tables applies.

## 6. Deprivation: English Indices of Deprivation 2025 (File 10)

Sheet `IMD` gives one row per local authority district, with headers on the first row.

In [14]:
imd = pd.read_excel(RAW + "File_10_IoD2025_Local_Authority_District_Summaries_lower-tier_v2.xlsx", sheet_name="IMD")
print(imd.shape)
imd.columns.tolist()

(296, 12)


['Local Authority District code (2024)',
 'Local Authority District name (2024)',
 'IMD - Average rank ',
 'IMD - Rank of average rank ',
 'IMD - Average score ',
 'IMD - Rank of average score ',
 'IMD - Proportion of LSOAs in most deprived 10% nationally ',
 'IMD - Rank of proportion of LSOAs in most deprived 10% nationally ',
 'IMD25 - Extent ',
 'IMD25 - Rank of extent ',
 'IMD25 - Local concentration ',
 'IMD25 - Rank of local concentration ']

**Observations**
- 296 local authorities, **England only**.
- `IMD - Rank of average score`: 1 = most deprived. This ranking is easy to explain, so it will be the main deprivation measure.

## 7. Income: gross disposable household income per head (ONS GDHI, Table 3)

Average income per person after taxes and benefits, in pounds, from 1997 to 2024. Headers are on row 4 (`header=3`).

In [15]:
gdhi = pd.read_excel(RAW + "ons_gdhi_local_authorities_1997to2024.xlsx", sheet_name="Table 3", header=3)
print(gdhi.shape)
# Year columns are stored as text, e.g. "2024"
gdhi[["Region", "LAD code", "LAD name", "2022", "2023", "2024"]].head()

(361, 31)


,Region,LAD code,LAD name,2022,2023,2024
0,North East,E06000001,Hartlepool,17957,19095,20080
1,North East,E06000004,Stockton-on-Tees,18477,19581,20623
2,North East,E06000002,Middlesbrough,16645,17134,17855
3,North East,E06000003,Redcar and Cleveland,17644,18672,19739
4,North East,E06000005,Darlington,19068,20457,21587


**Observations**
- 361 local authorities across the UK. The latest year is 2024, which will be used for the income analysis.

## 8. Do the area codes match across files?

Tables can only be joined if they use the same area codes. The charger table is used as the reference list.

In [16]:
charger_codes = set(evci["Local authority code"])

print("Charger codes missing from VEH0142:", sorted(charger_codes - set(veh0142["ONS Code"])))
print("Charger codes missing from GDHI:   ", sorted(charger_codes - set(gdhi["LAD code"])))

england_codes = {c for c in charger_codes if c.startswith("E")}
imd_codes = set(imd["Local Authority District code (2024)"])
print("England charger codes missing from IMD:", sorted(england_codes - imd_codes))
print("IMD codes missing from charger table:  ", sorted(imd_codes - england_codes))

Charger codes missing from VEH0142: ['E08000038', 'E08000039']
Charger codes missing from GDHI:    ['E08000038', 'E08000039']
England charger codes missing from IMD: ['E08000038', 'E08000039']
IMD codes missing from charger table:   ['E08000016', 'E08000019']


**Observations**
- All codes match apart from **Barnsley and Sheffield**. Their codes changed in 2024 after a small boundary change:
  - Barnsley: `E08000016` → `E08000038`
  - Sheffield: `E08000019` → `E08000039`
- The charger table uses the new codes and the other tables use the old ones. A small mapping will fix this during cleaning.

## Summary and next step

| Table | Sheet | Header row | Area code column | Unit | Format |
| --- | --- | --- | --- | --- | --- |
| VEH0105 | VEH0105 | 5 | `ONS Code` | thousands | wide (quarters) |
| VEH0142 | VEH0142 | 5 | `ONS Code` | number | wide (quarters) |
| VEH1153 | VEH1153a_RoadUsing | 5 | national only | thousands | long |
| EVCI0102 | EVCI0102a / b | 3 | `Local authority code` | number | 3 snapshots |
| EVCI9001 | 1a | 3 | `Local authority / region code` | number | wide (quarters) |
| IMD 2025 | IMD | 1 | `Local Authority District code (2024)` | rank / score | one row per area |
| GDHI | Table 3 | 4 | `LAD code` | £ per head | wide (years) |

**Next step (notebook 02):** clean the tables. Keep only lower-tier local authorities, align units, reshape to long format, fix the Barnsley/Sheffield codes, and save analysis-ready CSVs to `data/processed/`.